In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

In [ ]:
car_data = pd.read_csv('datasets/car_price_prediction.csv')

In [ ]:
car_data.head()

In [ ]:
car_data.shape

In [ ]:
car_data.columns

In [ ]:
car_data.info()

In [ ]:
car_data.describe()

In [ ]:
car_data.isnull().sum()

In [ ]:
car_data.duplicated().sum()

In [ ]:
car_data['Levy'].dtype

In [ ]:
type(car_data['Levy'].iloc[2])

In [ ]:
(car_data=='-').sum()

In [ ]:
car_data[car_data.duplicated()].head()

In [ ]:
car_data['Fuel type'].unique()

In [ ]:
print(car_data['Fuel type'].value_counts())
print(car_data['Category'].value_counts())
print(car_data['Gear box type'].value_counts())
print(car_data['Drive wheels'].value_counts())
print(car_data['Leather interior'].value_counts())
print(car_data['Wheel'].value_counts())
print(car_data['Color'].value_counts())
print(car_data['Doors'].value_counts())
print(car_data['Manufacturer'].value_counts())

In [ ]:
print(car_data['Prod. year'].min())
print(car_data['Prod. year'].max())

In [ ]:
plt.hist(car_data['Price'])
plt.xlabel('Price')
plt.ylabel('Number of cars')
plt.show()

In [ ]:
print(car_data['Category'].value_counts())
print(car_data['Gear box type'].value_counts())
print(car_data['Drive wheels'].value_counts())

In [ ]:
car_data['Prod. year'].value_counts().sort_index()

In [ ]:
car_data[['Price', 'Prod. year', 'Engine volume', 'Mileage', 'Cylinders', 'Airbags']].describe()

In [ ]:
car_data[['Price', 'Prod. year', 'Engine volume', 'Mileage',
          'Cylinders', 'Airbags', 'Levy']].dtypes

In [ ]:
car_data['Engine volume'].unique()[:30]

In [ ]:
car_data['Engine volume'].value_counts().head(20)

In [ ]:
print(car_data['Engine volume'].str.contains('Turbo').sum())
print(car_data['Engine volume'].str.contains('Turbo').value_counts())


In [ ]:
print(car_data['Mileage'].unique()[:20])
car_data['Mileage'].value_counts().head(20)

In [ ]:
car_data['Mileage'].str.contains('km').value_counts()

In [ ]:
print(car_data['Levy'].unique()[:20])
car_data['Levy'].value_counts().head(20)

In [ ]:
car_data['Levy'] = car_data['Levy'].replace('-', np.nan)

In [ ]:
print(car_data['Levy'].describe())
car_data['Levy'].isna().sum()

In [ ]:
# viable options to deal with missing data
# Impute/fill them — e.g. median.
# Drop the rows — probably undesirable here because that's ~30% of the dataset.
# Keep them as missing and use a preprocessing/model approach that can handle missing values.
# Potentially drop the Levy feature entirely if our analysis shows it isn't useful.

In [ ]:
# Converting the Levy column to numeric
car_data['Levy'] = pd.to_numeric(car_data['Levy'])
print(type(car_data['Levy']))
print(car_data['Levy'].dtype)

In [ ]:
car_data['Levy'].describe()

In [ ]:
car_data[['Levy', 'Price']].corr()

In [ ]:
car_data['Mileage'].dtype

In [ ]:
car_data['Mileage'].head()

In [ ]:
car_data['Mileage'] = car_data['Mileage'].str.replace(' km', '').astype(int)

In [ ]:
car_data['Engine volume'].head()

In [ ]:
car_data['Turbo'] = car_data['Engine volume'].str.contains('Turbo').astype(int)

In [ ]:
car_data[['Engine volume', 'Turbo']].head(10)

In [ ]:
car_data['Turbo'].value_counts()

In [ ]:
car_data['Engine volume'] = car_data['Engine volume'].str.replace('Turbo', '').astype(float)

In [ ]:
print(car_data['Engine volume'].dtype)
print(car_data['Turbo'].dtype)

In [ ]:
car_data.head()

In [ ]:
car_data.select_dtypes(include='object').columns

In [ ]:
cat_cols = car_data.select_dtypes(
    include=['object', 'string', 'category']
).columns

car_data[cat_cols].nunique().sort_values(ascending=False)

In [ ]:
car_data['Model'].value_counts().describe()

In [ ]:
car_data['Model'].value_counts().tail(20)

In [ ]:
manufacturer_counts = car_data['Manufacturer'].value_counts()

manufacturer_counts.describe()

In [ ]:
model_counts = car_data['Model'].value_counts()

for threshold in [2, 5, 10, 20, 50]:
    print(
        f"Threshold {threshold}: "
        f"{(model_counts >= threshold).sum()} categories kept"
    )

In [ ]:
for threshold in [2, 5, 10, 20, 50]:
    rows_kept = model_counts[model_counts >= threshold].sum()
    rows_other = len(car_data) - rows_kept
    
    print(
        f"Threshold {threshold}: "
        f"{rows_kept} rows kept, "
        f"{rows_other} rows → Other"
    )

In [ ]:
car_data['Levy'] = car_data['Levy'].fillna(car_data['Levy'].median())
print(car_data['Levy'].isna().sum())
print(car_data['Levy'].dtype)

In [ ]:
car_data.select_dtypes(include='number').columns

In [ ]:
car_data.select_dtypes(include='number').corr()['Price'].sort_values(ascending=False)

In [ ]:
car_data.groupby('Manufacturer')['Price'].median().sort_values(ascending=False)

In [ ]:
car_data.groupby('Model')['Price'].median().sort_values(ascending=False).head(20)

In [ ]:
car_data.groupby('Fuel type')['Price'].median().sort_values(ascending=False)

In [ ]:
car_data.groupby('Category')['Price'].median().sort_values(ascending=False)

In [ ]:
car_data.groupby('Gear box type')['Price'].median().sort_values(ascending=False)

In [ ]:
X = car_data.drop(['Price', 'ID'], axis=1)
y = car_data['Price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model_counts = X_train['Model'].value_counts()

rare_models = model_counts[model_counts < 10].index

X_train['Model'] = X_train['Model'].replace(rare_models, 'Other')
X_test['Model'] = X_test['Model'].replace(rare_models, 'Other')

In [ ]:
numeric_features = X_train.select_dtypes(include='number').columns
categorical_features = X_train.select_dtypes(exclude='number').columns

In [ ]:
numerical_processor = SimpleImputer(strategy='median')

categorical_processor = OneHotEncoder(handle_unknown='ignore')

In [ ]:
preprocessor = ColumnTransformer([
    ("numeric", numerical_processor, numeric_features),
    ("categorical", categorical_processor, categorical_features)
])

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [ ]:
print(X_train_processed.shape)
print(X_test_processed.shape)